FINBERT

In [1]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/

In [2]:
import mlflow
import mlflow.xgboost

In [3]:
mlflow.set_experiment(
    "finbert_xgboost_experiments"
)

2026/08/20 18:04:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/20 18:04:46 INFO mlflow.store.db.utils: Updating database tables
2026/08/20 18:04:53 INFO mlflow.tracking.fluent: Experiment with name 'finbert_xgboost_experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1787249093317, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787249093317, lifecycle_stage='active', name='finbert_xgboost_experiments', tags={}, trace_location=None, workspace='default'>

In [11]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from scipy.special import softmax


# =========================
# 1. LOAD DATA
# =========================

news_df = pd.read_csv("/content/drive/MyDrive/headlines.csv")

news_df["published_at"] = pd.to_datetime(
    news_df["published_at"],
    utc=True
)

news_df = news_df.sort_values(
    ["symbol", "published_at"]
)

# create date column

news_df["news_date"] = (
    news_df["published_at"]
    - pd.Timedelta(hours=9, minutes=15)
).dt.normalize()

print(news_df.shape)


# =========================
# 2. DEVICE
# =========================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", device)


# =========================
# 3. LOAD FINBERT
# =========================

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.to(device)

model.eval()


# =========================
# 4. FINBERT FUNCTION
# =========================

def run_finbert(texts):

    encoded = tokenizer(

        texts,

        return_tensors="pt",

        truncation=True,

        padding=True,

        max_length=128

    )

    encoded = {

        k: v.to(device)

        for k, v in encoded.items()

    }

    with torch.no_grad():

        outputs = model(

            **encoded,

            output_hidden_states=True

        )

    probs = softmax(

        outputs.logits.cpu().numpy(),

        axis=1

    )

    embeddings = (

        outputs.hidden_states[-1]

        [:, 0, :]

        .cpu()

        .numpy()

    )

    return probs, embeddings


# =========================
# 5. RUN FINBERT
# =========================

batch_size = 32

all_probs = []

all_embs = []

for i in range(

    0,

    len(news_df),

    batch_size

):

    batch_text = (

        news_df["headline"]

        .iloc[i:i+batch_size]

        .astype(str)

        .tolist()

    )

    probs, embs = run_finbert(

        batch_text

    )

    all_probs.append(

        probs

    )

    all_embs.append(

        embs

    )

    print(

        f"Processed {i+len(batch_text)}/{len(news_df)}"

    )


all_probs = np.vstack(

    all_probs

)

all_embs = np.vstack(

    all_embs

)


# =========================
# 6. ATTACH RESULTS
# =========================

#news_df["positive_prob"] = (

 #   all_probs[:,0]

#)

#news_df["negative_prob"] = (

#    all_probs[:,1]

#)

#news_df["neutral_prob"] = (

 #   all_probs[:,2]

#)
print(model.config.id2label)
label_map = model.config.id2label

for idx, label in label_map.items():

    news_df[f"{label}_prob"] = all_probs[:, idx]

news_df["sentiment_score"] = (

    news_df["positive_prob"]

    -

    news_df["negative_prob"]

) * (

    1

    -

    news_df["neutral_prob"]

)


news_df["sentiment_label"] = np.where(

    news_df["sentiment_score"] > 0.1,

    "positive",

    np.where(

        news_df["sentiment_score"] < -0.1,

        "negative",

        "neutral"

    )

)

news_df["embedding"] = list(

    all_embs

)



(8827, 8)
Using: cpu


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Processed 32/8827
Processed 64/8827
Processed 96/8827
Processed 128/8827
Processed 160/8827
Processed 192/8827
Processed 224/8827
Processed 256/8827
Processed 288/8827
Processed 320/8827
Processed 352/8827
Processed 384/8827
Processed 416/8827
Processed 448/8827
Processed 480/8827
Processed 512/8827
Processed 544/8827
Processed 576/8827
Processed 608/8827
Processed 640/8827
Processed 672/8827
Processed 704/8827
Processed 736/8827
Processed 768/8827
Processed 800/8827
Processed 832/8827
Processed 864/8827
Processed 896/8827
Processed 928/8827
Processed 960/8827
Processed 992/8827
Processed 1024/8827
Processed 1056/8827
Processed 1088/8827
Processed 1120/8827
Processed 1152/8827
Processed 1184/8827
Processed 1216/8827
Processed 1248/8827
Processed 1280/8827
Processed 1312/8827
Processed 1344/8827
Processed 1376/8827
Processed 1408/8827
Processed 1440/8827
Processed 1472/8827
Processed 1504/8827
Processed 1536/8827
Processed 1568/8827
Processed 1600/8827
Processed 1632/8827
Processed 1664

KeyboardInterrupt: 

['/content/drive/MyDrive/finbert_pca.pkl']

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Faeture Engineering


In [4]:
import pandas as pd
import numpy as np

ohlcv = pd.read_csv("/content/drive/MyDrive/merged_ohlc_15min.csv")

ohlcv["datetime"] = pd.to_datetime(
    ohlcv["datetime"]
)

ohlcv = ohlcv.sort_values(
    ["symbol","datetime"]
)

ohlcv["date"] = (
    ohlcv["datetime"]
    .dt.normalize()
)
ohlcv = ohlcv.sort_values(
    ["symbol", "date", "datetime"]
)

In [5]:
#Creating daily ohlcv
daily_ohlcv = (

    ohlcv

    .groupby(

        ["symbol","date"]

    )

    .agg(

        day_open=("open","first"),

        day_high=("high","max"),

        day_low=("low","min"),

        day_close=("close","last"),

        day_volume=("volume","sum")

    )

    .reset_index()

)
daily_ohlcv = daily_ohlcv.sort_values(

    ["symbol","date"]

)

In [6]:
#historical ohlcv
g = daily_ohlcv.groupby("symbol")

#Daily return
daily_ohlcv["daily_return"] = (

    g["day_close"]

    .pct_change()

)

#intra day movement
daily_ohlcv["open_close_pct"] = (

    daily_ohlcv["day_close"]

    - daily_ohlcv["day_open"]

) / daily_ohlcv["day_open"]


#Daily Range
daily_ohlcv["high_low_pct"] = (

    daily_ohlcv["day_high"]

    - daily_ohlcv["day_low"]

) / daily_ohlcv["day_low"]

#LAg features
for lag in [1,2,3]:

    daily_ohlcv[f"return_lag_{lag}"] = (

        g["daily_return"]

        .shift(lag)

    )

#Rolling Returns

for w in [7,14,30]:

    daily_ohlcv[f"return_{w}d"] = (

        g["daily_return"]

        .transform(

            lambda x:

            x.rolling(w)

            .mean()

        )

    )

#Volatility
for w in [7,14,30]:

    daily_ohlcv[f"volatility_{w}d"] = (

        g["daily_return"]

        .transform(

            lambda x:

            x.rolling(w)

            .std()

        )

    )

#Moving Averages
for w in [7,14,30]:

    daily_ohlcv[f"price_ma_{w}"] = (

        g["day_close"]

        .transform(

            lambda x:

            x.rolling(w)

            .mean()

        )

    )

#price versus moving averages
for w in [7,14,30]:

    daily_ohlcv[f"price_vs_ma{w}"] = (

        daily_ohlcv["day_close"]

        - daily_ohlcv[f"price_ma_{w}"]

    ) / daily_ohlcv[f"price_ma_{w}"]


#volume features
for w in [7,14,30]:

    daily_ohlcv[f"volume_{w}d"] = (

        g["day_volume"]

        .transform(

            lambda x:

            x.rolling(w)

            .mean()

        )

    )
daily_ohlcv["volume_ratio"] = (

    daily_ohlcv["day_volume"]

    / daily_ohlcv["volume_7d"]

)


In [7]:
#extract d+1 9:15
open_915 = (

    ohlcv

    .sort_values(

        ["symbol","datetime"]

    )

    .groupby(

        ["symbol","date"]

    )

    .first()

    .reset_index()

)
open_915 = open_915[

    [

    "symbol",

    "date",

    "open",

    "close"

    ]

]

open_915 = open_915.rename(

    columns={

        "open":"open_915",

        "close":"close_915"

    }

)

#shift 9:15 packet backward
open_915["date"] = (

    open_915["date"]

    - pd.Timedelta(days=1)

)

#Attach and merge with d day data
open_915["date"] = (

    open_915["date"]

    - pd.Timedelta(days=1)

)
daily_ohlcv = daily_ohlcv.merge(

    open_915,

    on=["symbol","date"],

    how="left"

)

#9:15 features
#over night gap
daily_ohlcv["gap_from_prev_close"] = (

    daily_ohlcv["open_915"]

    - daily_ohlcv["day_close"]

) / daily_ohlcv["day_close"]

#first 15min return
daily_ohlcv["first15_return"] = (

    daily_ohlcv["close_915"]

    - daily_ohlcv["open_915"]

) / daily_ohlcv["open_915"]

#15min direction
daily_ohlcv["first15_direction"] = np.sign(

    daily_ohlcv["first15_return"]

)




In [8]:
#Creating target for training
daily_ohlcv["target_return"] = (

    daily_ohlcv["day_close"]

    .shift(-1)

    - daily_ohlcv["open_915"]

) / daily_ohlcv["open_915"]

daily_ohlcv["target_return"] = (

    daily_ohlcv

    .groupby("symbol")

    .apply(

        lambda x:

        (

            x["day_close"]

            .shift(-1)

            - x["open_915"]

        )

        / x["open_915"]

    )

    .reset_index(level=0,drop=True)

)

/tmp/ipykernel_5107/3811936642.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [9]:
#Create classes

daily_ohlcv["target"] = np.select(

    [

        daily_ohlcv["target_return"] < -0.003,

        daily_ohlcv["target_return"] > 0.003

    ],

    [

        0,

        2

    ],

    default=1

)

In [10]:
from sklearn.decomposition import PCA
import joblib
# =========================
# REDUCE EMBEDDINGS
# =========================

PCA_COMPONENTS = 50
pca = PCA(
    n_components=PCA_COMPONENTS,
    random_state=42
)
reduced_embs = pca.fit_transform(
  all_embs
)
joblib.dump(
    pca,
    "/content/drive/MyDrive/finbert_pca.pkl"
)
#pca = joblib.load(
#    "finbert_pca.pkl"
#)

#new_embeddings = pca.transform(
 #   all_embs
#)
news_df["embedding"] = list(
    reduced_embs
    #new_embeddings
)

print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)


news_df["published_at"] = (
    pd.to_datetime(news_df["published_at"], utc=True)
    .dt.tz_localize(None)
)

news_df["news_date"] = (
    news_df["published_at"]
    .dt.normalize()
)

# Stock dates
daily_ohlcv["date"] = (
    pd.to_datetime(daily_ohlcv["date"])
    .dt.tz_localize(None)
    .dt.normalize()
)


# =========================
# 7-DAY AGGREGATION USING STOCK DATES
# =========================

final_rows = []

# Daily stock dates
stock_dates = (
    daily_ohlcv[["symbol", "date"]]
    .drop_duplicates()
    .sort_values(["symbol", "date"])
)

for symbol, stock_group in stock_dates.groupby("symbol"):

    # All news for this symbol
    news_group = news_df[
        news_df["symbol"] == symbol
    ].copy()

    news_group = news_group.sort_values(
        "published_at"
    )

    for current_date in stock_group["date"]:

        start_date = current_date - pd.Timedelta(days=6)

        # Previous 7 calendar days (including current day)
        window = news_group[

            (news_group["news_date"] >= start_date)

            &

            (news_group["news_date"] <= current_date)

        ].copy()

        # No news in last 7 days
        if len(window) == 0:

            row = {

                "symbol": symbol,

                "date": current_date,

                "article_count": 0,

                "weighted_sentiment": 0,

                "sentiment_std": 0,

                "sentiment_max": 0,

                "sentiment_min": 0,

                "positive_articles": 0,

                "negative_articles": 0,

                "neutral_articles": 0,

                "avg_positive_prob": 0,

                "avg_negative_prob": 0,

                "avg_neutral_prob": 0,

                "sentiment_trend": 0

            }

            # Zero embeddings
            for i in range(PCA_COMPONENTS):

                row[f"emb_{i}"] = 0

            final_rows.append(row)

            continue

        # =====================
        # Recency weights
        # =====================

        latest_time = window["published_at"].max()

        window["age_days"] = (

            latest_time

            - window["published_at"]

        ).dt.total_seconds() / 86400

        lambda_decay = 0.5

        window["weight"] = np.exp(

            -lambda_decay * window["age_days"]

        )

        window["weight"] /= window["weight"].sum()

        weights = window["weight"].values

        # =====================
        # Weighted Sentiment
        # =====================

        weighted_sentiment = np.average(

            window["sentiment_score"],

            weights=weights

        )

        avg_positive = np.average(

            window["positive_prob"],

            weights=weights

        )

        avg_negative = np.average(

            window["negative_prob"],

            weights=weights

        )

        avg_neutral = np.average(

            window["neutral_prob"],

            weights=weights

        )

        # =====================
        # Weighted Embeddings
        # =====================

        emb = np.vstack(

            window["embedding"]

        )

        daily_embedding = np.average(

            emb,

            axis=0,

            weights=weights

        )

        # =====================
        # Sentiment Trend
        # =====================

        sentiment_trend = (

            window

            .sort_values("published_at")

            ["sentiment_score"]

            .tail(3)

            .mean()

            -

            window

            .sort_values("published_at")

            ["sentiment_score"]

            .head(3)

            .mean()

        )

        row = {

            "symbol": symbol,

            "date": current_date,

            "article_count": len(window),

            "weighted_sentiment": weighted_sentiment,

            "sentiment_std": window["sentiment_score"].std(),

            "sentiment_max": window["sentiment_score"].max(),

            "sentiment_min": window["sentiment_score"].min(),

            "positive_articles": (

                window["sentiment_label"] == "positive"

            ).sum(),

            "negative_articles": (

                window["sentiment_label"] == "negative"

            ).sum(),

            "neutral_articles": (

                window["sentiment_label"] == "neutral"

            ).sum(),

            "avg_positive_prob": avg_positive,

            "avg_negative_prob": avg_negative,

            "avg_neutral_prob": avg_neutral,

            "sentiment_trend": sentiment_trend

        }

        for i in range(PCA_COMPONENTS):

            row[f"emb_{i}"] = daily_embedding[i]

        final_rows.append(row)

# Final FinBERT output
final_df = pd.DataFrame(final_rows)

NameError: name 'all_embs' is not defined

In [ ]:
final_df.to_csv(

    "/content/drive/MyDrive/news_features.csv",

    index=False

)

print(

    final_df.head()

)

print(

    final_df.shape
)

              symbol       date  article_count  weighted_sentiment  \
0  Adani Enterprises 2025-08-01              0                 0.0   
1  Adani Enterprises 2025-08-04              0                 0.0   
2  Adani Enterprises 2025-08-05              0                 0.0   
3  Adani Enterprises 2025-08-06              0                 0.0   
4  Adani Enterprises 2025-08-07              0                 0.0   

   sentiment_std  sentiment_max  sentiment_min  positive_articles  \
0            0.0            0.0            0.0                  0   
1            0.0            0.0            0.0                  0   
2            0.0            0.0            0.0                  0   
3            0.0            0.0            0.0                  0   
4            0.0            0.0            0.0                  0   

   negative_articles  neutral_articles  ...  emb_40  emb_41  emb_42  emb_43  \
0                  0                 0  ...     0.0     0.0     0.0     0.0   
1     

In [10]:
#Merge ohlcv and finbert
#for now finbert output save in a file we create df on that file

import pandas as pd

# =========================
# LOAD FINBERT CSV
# =========================

finbert_df = pd.read_csv(
    "/content/drive/MyDrive/news_features.csv"
)

# =========================
# CONVERT DATE COLUMNS
# =========================

finbert_df["date"] = (

    pd.to_datetime(

        finbert_df["date"],

        utc=True

    )

    .dt.tz_localize(None)

    .dt.normalize()

)

daily_ohlcv["date"] = (

    pd.to_datetime(

        daily_ohlcv["date"]

    )

    .dt.normalize()

)
# =========================
# MERGE
# =========================

master_df = finbert_df.merge(

    daily_ohlcv,

    on=["symbol", "date"],

    how="inner"

)

print(master_df.shape)

print(master_df.head())


(10896, 98)
              symbol       date  article_count  weighted_sentiment  \
0  Adani Enterprises 2025-08-01              0                 0.0   
1  Adani Enterprises 2025-08-04              0                 0.0   
2  Adani Enterprises 2025-08-05              0                 0.0   
3  Adani Enterprises 2025-08-06              0                 0.0   
4  Adani Enterprises 2025-08-07              0                 0.0   

   sentiment_std  sentiment_max  sentiment_min  positive_articles  \
0            0.0            0.0            0.0                  0   
1            0.0            0.0            0.0                  0   
2            0.0            0.0            0.0                  0   
3            0.0            0.0            0.0                  0   
4            0.0            0.0            0.0                  0   

   negative_articles  neutral_articles  ...  volume_14d  volume_30d  \
0                  0                 0  ...         NaN         NaN   
1         

In [13]:
#Merged csv

master_df.to_csv(

    "/content/drive/MyDrive/merged_features.csv",

    index=False

)

XG BOOST Model Training

In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (

    accuracy_score,

    classification_report,

    confusion_matrix

)

from xgboost import XGBClassifier

import joblib

In [13]:

merged_df = master_df.copy()

merged_df = merged_df.sort_values(

    ["date","symbol"]

)
#encode symbol
le = LabelEncoder()

merged_df["symbol_encoded"] = (

    le.fit_transform(

        merged_df["symbol"]

    )

)
joblib.dump(
    le,
    "/content/drive/MyDrive/symbol_encoder.pkl"
)
#le = joblib.load("symbol_encoder.pkl")

#prediction_df["symbol_encoded"] = le.transform(
#    prediction_df["symbol"]
#)

#Handle nulls
merged_df = merged_df.replace(

    [np.inf,-np.inf],

    np.nan

)

merged_df = merged_df.sort_values(["date", "symbol"])

# Features
X = merged_df.drop(
    columns=[
        "symbol",
        "date",
        "target",
        "target_return"
    ]
)
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "/content/drive/MyDrive/feature_columns.pkl"
)

# Target
y = merged_df["target"]



In [18]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

import pandas as pd
import numpy as np

import mlflow
import mlflow.xgboost


# ============================================
# CREATE MLFLOW EXPERIMENT
# ============================================

mlflow.set_experiment(
    "finbert_xgboost_experiments"
)


# ============================================
# PARAMETER COMBINATIONS TO TEST
# ============================================

parameter_sets = [

    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

]


# ============================================
# TIME SERIES SPLIT
# ============================================

tscv = TimeSeriesSplit(
    n_splits=5
)


best_f1 = -1

best_accuracy = -1

best_params = None

best_predictions_df = None


# ============================================
# RUN DIFFERENT EXPERIMENTS
# ============================================

for experiment_number, params in enumerate(
    parameter_sets,
    start=1
):

    accuracy_scores = []

    f1_scores = []

    all_predictions = []


    print(
        "\n===================================="
    )

    print(
        f"Experiment {experiment_number}"
    )

    print(
        params
    )

    print(
        "===================================="
    )


    # One parameter set = one MLflow run

    with mlflow.start_run(
        run_name=f"xgboost_experiment_{experiment_number}"
    ):


        # ========================================
        # LOG PIPELINE PARAMETERS
        # ========================================

        mlflow.log_param(
            "finbert_model",
            "ProsusAI/finbert"
        )

        mlflow.log_param(
            "pca_components",
            50
        )

        mlflow.log_param(
            "news_window_days",
            7
        )

        mlflow.log_param(
            "time_series_splits",
            5
        )


        # XGBoost parameters

        mlflow.log_params(
            params
        )


        # ========================================
        # TIME SERIES CROSS VALIDATION
        # ========================================

        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            tscv.split(X),
            start=1
        ):


            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]


            y_train = y.iloc[
                train_idx
            ]

            y_test = y.iloc[
                test_idx
            ]


            # ====================================
            # CREATE MODEL
            # ====================================

            model = XGBClassifier(

                objective="multi:softprob",

                num_class=3,

                random_state=42,

                eval_metric="mlogloss",

                **params
            )


            # ====================================
            # TRAIN MODEL
            # ====================================

            model.fit(
                X_train,
                y_train
            )


            # ====================================
            # PREDICTIONS
            # ====================================

            y_pred = model.predict(
                X_test
            )


            y_prob = model.predict_proba(
                X_test
            )


            # ====================================
            # METRICS
            # ====================================

            acc = accuracy_score(
                y_test,
                y_pred
            )


            f1 = f1_score(
                y_test,
                y_pred,
                average="macro"
            )


            accuracy_scores.append(
                acc
            )

            f1_scores.append(
                f1
            )


            # ====================================
            # LOG FOLD METRICS TO MLFLOW
            # ====================================

            mlflow.log_metric(
                f"fold_{fold}_accuracy",
                acc
            )


            mlflow.log_metric(
                f"fold_{fold}_macro_f1",
                f1
            )


            print(
                f"Fold {fold}"
            )

            print(
                f"Accuracy : {acc:.4f}"
            )

            print(
                f"Macro F1 : {f1:.4f}"
            )

            print(
                "-" * 40
            )


            # ====================================
            # YOUR EXISTING PREDICTION DATAFRAME
            # ====================================

            fold_predictions = merged_df.iloc[
                test_idx
            ][
                [
                    "symbol",
                    "date",
                    "target_return"
                ]
            ].copy()


            fold_predictions[
                "Actual"
            ] = y_test.values


            fold_predictions[
                "Predicted"
            ] = y_pred


            fold_predictions[
                "Prob_Negative"
            ] = y_prob[:, 0]


            fold_predictions[
                "Prob_Neutral"
            ] = y_prob[:, 1]


            fold_predictions[
                "Prob_Positive"
            ] = y_prob[:, 2]


            fold_predictions[
                "Confidence"
            ] = y_prob.max(
                axis=1
            )


            fold_predictions[
                "Correct"
            ] = (
                fold_predictions[
                    "Actual"
                ]
                ==
                fold_predictions[
                    "Predicted"
                ]
            )


            fold_predictions[
                "Fold"
            ] = fold


            fold_predictions[
                "Experiment"
            ] = experiment_number


            all_predictions.append(
                fold_predictions
            )


        # ========================================
        # COMBINE ALL 5 FOLDS
        # ========================================

        predictions_df = pd.concat(
            all_predictions,
            ignore_index=True
        )


        # ========================================
        # AVERAGE METRICS
        # ========================================

        avg_accuracy = np.mean(
            accuracy_scores
        )


        avg_f1 = np.mean(
            f1_scores
        )


        print(
            "Average Accuracy:",
            avg_accuracy
        )


        print(
            "Average Macro F1:",
            avg_f1
        )


        # ========================================
        # LOG AVERAGE METRICS
        # ========================================

        mlflow.log_metric(
            "average_accuracy",
            avg_accuracy
        )


        mlflow.log_metric(
            "average_macro_f1",
            avg_f1
        )


        # ========================================
        # SAVE THIS EXPERIMENT PREDICTIONS
        # ========================================

        prediction_file = (
            f"/content/drive/MyDrive/"
            f"timeseries_predictions_"
            f"experiment_{experiment_number}.csv"
        )


        predictions_df.to_csv(
            prediction_file,
            index=False
        )


        # Also log CSV into MLflow

        mlflow.log_artifact(
            prediction_file
        )


        # ========================================
        # CHECK IF THIS IS BEST MODEL
        # ========================================

        if avg_f1 > best_f1:

            best_f1 = avg_f1

            best_accuracy = avg_accuracy

            best_params = params.copy()

            best_predictions_df = (
                predictions_df.copy()
            )


Experiment 1
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
Fold 1
Accuracy : 0.7869
Macro F1 : 0.3896
----------------------------------------
Fold 2
Accuracy : 0.7847
Macro F1 : 0.4095
----------------------------------------
Fold 3
Accuracy : 0.6685
Macro F1 : 0.4486
----------------------------------------
Fold 4
Accuracy : 0.6779
Macro F1 : 0.5353
----------------------------------------
Fold 5
Accuracy : 0.7120
Macro F1 : 0.5758
----------------------------------------
Average Accuracy: 0.7259911894273128
Average Macro F1: 0.47175064542231115

Experiment 2
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8}
Fold 1
Accuracy : 0.7891
Macro F1 : 0.3885
----------------------------------------
Fold 2
Accuracy : 0.7742
Macro F1 : 0.3910
----------------------------------------
Fold 3
Accuracy : 0.6542
Macro F1 : 0.4031
----------------------------------------
Fold 4
Accuracy :

In [19]:
print(
    "\n===================================="
)

print(
    "BEST EXPERIMENT"
)

print(
    "===================================="
)

print(
    "Best Parameters:"
)

print(
    best_params
)

print(
    "Best Average Accuracy:",
    best_accuracy
)

print(
    "Best Average Macro F1:",
    best_f1
)


BEST EXPERIMENT
Best Parameters:
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
Best Average Accuracy: 0.7259911894273128
Best Average Macro F1: 0.47175064542231115


In [20]:
from xgboost import XGBClassifier

final_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

final_model.fit(
    X,
    y
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=3, ...)

In [ ]:
joblib.dump(
    final_model,
    "/content/drive/MyDrive/xgboost_model.pkl"
)

['/content/drive/MyDrive/xgboost_model.pkl']

In [1]:
!pip install lightgbm mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [2]:
# ============================================================
# IMPORTS
# ============================================================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score

from lightgbm import LGBMClassifier

import pandas as pd
import numpy as np
import joblib

import mlflow
import mlflow.lightgbm

In [14]:
mlflow.set_experiment(
    "lightgbm_experiments"
)


2026/08/21 18:36:41 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/21 18:36:41 INFO mlflow.store.db.utils: Updating database tables
2026/08/21 18:36:45 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1787337405725, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787337405725, lifecycle_stage='active', name='lightgbm_experiments', tags={}, trace_location=None, workspace='default'>

In [15]:
# ============================================================
# LIGHTGBM PARAMETER SETS
# ============================================================

parameter_sets = [

    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },

    {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.05,
        "num_leaves": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }

]
# ============================================================
# TIME SERIES CROSS VALIDATION
# ============================================================

tscv = TimeSeriesSplit(
    n_splits=5
)


In [16]:
# ============================================================
# BEST MODEL TRACKING
# ============================================================

best_f1 = -1

best_accuracy = -1

best_params = None

best_predictions_df = None

best_experiment_number = None

# ============================================================
# LIGHTGBM EXPERIMENT LOOP
# ============================================================

for experiment_number, params in enumerate(
    parameter_sets,
    start=1
):

    accuracy_scores = []

    f1_scores = []

    all_predictions = []


    print(
        "\n=============================================="
    )

    print(
        f"LIGHTGBM EXPERIMENT {experiment_number}"
    )

    print(
        "Parameters:"
    )

    print(
        params
    )

    print(
        "=============================================="
    )


    # ========================================================
    # ONE PARAMETER SET = ONE MLFLOW RUN
    # ========================================================

    with mlflow.start_run(
        run_name=f"lightgbm_experiment_{experiment_number}"
    ):


        # ====================================================
        # LOG GENERAL PIPELINE PARAMETERS
        # ====================================================

        mlflow.log_param(
            "model_type",
            "LightGBM"
        )

        mlflow.log_param(
            "finbert_model",
            "ProsusAI/finbert"
        )

        mlflow.log_param(
            "pca_components",
            50
        )

        mlflow.log_param(
            "news_window_days",
            7
        )

        mlflow.log_param(
            "time_series_splits",
            5
        )


        # ====================================================
        # LOG LIGHTGBM PARAMETERS
        # ====================================================

        mlflow.log_params(
            params
        )


        # ====================================================
        # 5-FOLD TIME SERIES VALIDATION
        # ====================================================

        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            tscv.split(X),
            start=1
        ):


            # ================================================
            # TRAIN / TEST DATA
            # ================================================

            X_train = X.iloc[
                train_idx
            ]

            X_test = X.iloc[
                test_idx
            ]


            y_train = y.iloc[
                train_idx
            ]

            y_test = y.iloc[
                test_idx
            ]


            # ================================================
            # CREATE LIGHTGBM MODEL
            # ================================================

            model = LGBMClassifier(

                objective="multiclass",

                num_class=3,

                random_state=42,

                verbosity=-1,

                **params
            )


            # ================================================
            # TRAIN MODEL
            # ================================================

            model.fit(
                X_train,
                y_train
            )


            # ================================================
            # PREDICTIONS
            # ================================================

            y_pred = model.predict(
                X_test
            )


            y_prob = model.predict_proba(
                X_test
            )


            # ================================================
            # METRICS
            # ================================================

            acc = accuracy_score(
                y_test,
                y_pred
            )


            f1 = f1_score(
                y_test,
                y_pred,
                average="macro"
            )


            accuracy_scores.append(
                acc
            )

            f1_scores.append(
                f1
            )


            # ================================================
            # LOG FOLD METRICS IN MLFLOW
            # ================================================

            mlflow.log_metric(
                f"fold_{fold}_accuracy",
                acc
            )


            mlflow.log_metric(
                f"fold_{fold}_macro_f1",
                f1
            )


            # ================================================
            # PRINT FOLD RESULTS
            # ================================================

            print(
                f"Fold {fold}"
            )

            print(
                f"Accuracy : {acc:.4f}"
            )

            print(
                f"Macro F1 : {f1:.4f}"
            )

            print(
                "-" * 40
            )


            # ================================================
            # CREATE PREDICTION DATAFRAME
            #
            # SAME STYLE AS YOUR XGBOOST OUTPUT
            # ================================================

            fold_predictions = merged_df.iloc[
                test_idx
            ][
                [
                    "symbol",
                    "date",
                    "target_return"
                ]
            ].copy()


            fold_predictions[
                "Actual"
            ] = y_test.values


            fold_predictions[
                "Predicted"
            ] = y_pred


            # IMPORTANT:
            # These assume classes are 0,1,2
            # in the same order as your XGBoost target.
            fold_predictions[
                "Prob_Negative"
            ] = y_prob[:, 0]


            fold_predictions[
                "Prob_Neutral"
            ] = y_prob[:, 1]


            fold_predictions[
                "Prob_Positive"
            ] = y_prob[:, 2]


            fold_predictions[
                "Confidence"
            ] = y_prob.max(
                axis=1
            )


            fold_predictions[
                "Correct"
            ] = (
                fold_predictions[
                    "Actual"
                ]
                ==
                fold_predictions[
                    "Predicted"
                ]
            )


            fold_predictions[
                "Fold"
            ] = fold


            fold_predictions[
                "Experiment"
            ] = experiment_number


            fold_predictions[
                "Model"
            ] = "LightGBM"


            all_predictions.append(
                fold_predictions
            )


        # ====================================================
        # COMBINE ALL 5 FOLDS
        # ====================================================

        predictions_df = pd.concat(
            all_predictions,
            ignore_index=True
        )


        # ====================================================
        # AVERAGE METRICS
        # ====================================================

        avg_accuracy = np.mean(
            accuracy_scores
        )


        avg_f1 = np.mean(
            f1_scores
        )


        print(
            "\nAverage Accuracy:",
            avg_accuracy
        )


        print(
            "Average Macro F1:",
            avg_f1
        )


        # ====================================================
        # LOG AVERAGE METRICS TO MLFLOW
        # ====================================================

        mlflow.log_metric(
            "average_accuracy",
            avg_accuracy
        )


        mlflow.log_metric(
            "average_macro_f1",
            avg_f1
        )


        # ====================================================
        # SAVE THIS EXPERIMENT'S PREDICTIONS
        # ====================================================

        prediction_file = (
            "/content/drive/MyDrive/"
            f"lightgbm_predictions_experiment_"
            f"{experiment_number}.csv"
        )


        predictions_df.to_csv(
            prediction_file,
            index=False
        )


        # ====================================================
        # LOG PREDICTION FILE IN MLFLOW
        # ====================================================

        mlflow.log_artifact(
            prediction_file
        )


        # ====================================================
        # CHECK WHETHER THIS IS THE BEST EXPERIMENT
        #
        # PRIMARY METRIC = MACRO F1
        # ====================================================

        if avg_f1 > best_f1:

            best_f1 = avg_f1

            best_accuracy = avg_accuracy

            best_params = params.copy()

            best_predictions_df = (
                predictions_df.copy()
            )

            best_experiment_number = (
                experiment_number
            )
# ============================================================
# DISPLAY BEST LIGHTGBM EXPERIMENT
# ============================================================

print(
    "\n=============================================="
)

print(
    "BEST LIGHTGBM EXPERIMENT"
)

print(
    "=============================================="
)


print(
    "Experiment Number:",
    best_experiment_number
)


print(
    "\nBest Parameters:"
)


print(
    best_params
)


print(
    "\nBest Average Accuracy:",
    best_accuracy
)


print(
    "Best Average Macro F1:",
    best_f1
)


LIGHTGBM EXPERIMENT 1
Parameters:
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'num_leaves': 15, 'subsample': 0.8, 'colsample_bytree': 0.8}
Fold 1
Accuracy : 0.7781
Macro F1 : 0.4020
----------------------------------------
Fold 2
Accuracy : 0.7709
Macro F1 : 0.3987
----------------------------------------
Fold 3
Accuracy : 0.6674
Macro F1 : 0.4556
----------------------------------------
Fold 4
Accuracy : 0.6801
Macro F1 : 0.5369
----------------------------------------
Fold 5
Accuracy : 0.7015
Macro F1 : 0.5592
----------------------------------------

Average Accuracy: 0.7196035242290748
Average Macro F1: 0.47048154876298903

LIGHTGBM EXPERIMENT 2
Parameters:
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.03, 'num_leaves': 31, 'subsample': 0.8, 'colsample_bytree': 0.8}
Fold 1
Accuracy : 0.7825
Macro F1 : 0.4060
----------------------------------------
Fold 2
Accuracy : 0.7638
Macro F1 : 0.3998
----------------------------------------
Fold 3
Accuracy : 0.66

In [20]:
import joblib
# ============================================================
# FINAL LIGHTGBM MODEL
# ============================================================

final_lgbm_model = LGBMClassifier(

    objective="multiclass",

    num_class=3,

    random_state=42,

    verbosity=-1,

    n_estimators= 300,
    max_depth= 4,
    learning_rate= 0.05,
    num_leaves= 15,
    subsample= 0.8,
    colsample_bytree= 0.8
)
final_lgbm_model.fit(
    X,
    y
)


# ============================================================
# SAVE FINAL MODEL AS PKL
# ============================================================

lightgbm_model_path = (
    "/content/drive/MyDrive/"
    "lightgbm_model.pkl"
)

joblib.dump(
    final_lgbm_model,
    lightgbm_model_path
)

print("Final LightGBM model trained successfully")
print("Model saved at:")
print(lightgbm_model_path)

Final LightGBM model trained successfully
Model saved at:
/content/drive/MyDrive/lightgbm_model.pkl
